In [1]:
import numpy as np
from matplotlib import pyplot as plt

from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
import menstrualmodel as mm

In [4]:
# Define constants used in equations.
a_1, a_2 = 0, 0
b_1, b_2 = 0.02, 0.9
s_1, s_2 = 2, 1.5
mu = 0.002
k = 0.000025    # The paper uses 0.00025, but it's hard to determine which solution actually makes sense.
g = 30
c = 0.007
B_1, B_2 = 14, 1
A_1, A_2 = 250000, 75

c1, c2, c3, c4, c5 = 1, 1, 1, 1, 1

constants = c1, c2, c3, c4, c5

# Define other constants.
G0, L0, E0 = 1.0, .25, 1.0
n = 2000
t_f = 3*28

initial_conditions = {
    'GnRH': 1.0,
    'LH': 0.25,
    'Estrogen': 1.0
    }


model = mm.ControlMenstrualModel(initial_hormones=initial_conditions, time_domain=(0, 3*28), resolution=1000)
sol = model.simulate([0, 0, 0])
model.plot(sol, plot_day_14=True, ylim=(0, 12), title="Best Fit Model")
plt.show()

TypeError: 'list' object is not callable

In [11]:
# Initialize state, costate, and u.
state0 = np.array([G0, L0, E0])
costate0 = np.zeros(3)

u = np.zeros((3, n))
u[0], u[1] = b_1, b_2

max_step = 0.5

epsilon = 0.001
test = epsilon + 1

tls = np.linspace(0, t_f, n)
while(test > epsilon):
    oldu = u.copy()
    u_interpolation = CubicSpline(tls, oldu, axis=1)

    # Solve the state equations forward in time.
    state_solution = solve_ivp(
        model.ode, 
        t_span=(0, t_f), 
        y0=state0, 
        method='RK45',
        t_eval=tls,
        args=(u_interpolation,),
        dense_output=True,
        max_step=max_step
        )

    # Solve the costate equations backward in time.
    costate_solution = solve_ivp(
        model.costate_equations,
        t_span=(t_f, 0),
        y0=costate0,
        method='RK45',
        t_eval=tls[::-1],
        args=(u_interpolation, state_solution, constants),
        dense_output=True,
        max_step=max_step
        )

    # Solve for u1 and u2.
    G, L, E = state_solution.y
    lam1, lam2, lam3 = costate_solution.y[:, ::-1]
    print(lam1.shape)
    print(lam2.shape)
    print(lam3.shape)

    uG = lam1 / (2 * c3) 
    uL = lam2 / (2 * c5)
    uE = lam3 / (2 * c4)
    
    # Update control u with u1 and u2.
    u = np.array([uG, uL, uE])

    # Test for convergence
    test = abs(oldu - u).sum()

(2000,)
(2000,)
(2000,)
(1756,)
(1756,)
(1756,)


ValueError: operands could not be broadcast together with shapes (3,2000) (3,1756) 

In [ ]:
plt.subplot(2, 3, 1)
plt.plot(tls, G)
plt.xlabel('Days')
plt.ylabel('Concentration')
plt.title('T')

plt.subplot(2, 3, 2)
plt.plot(tls, L)
plt.xlabel('Days')
plt.ylabel('Concentration')
plt.title('L')

plt.subplot(2, 3, 3)
plt.plot(tls, E)
plt.xlabel('Days')
plt.ylabel('Control')
plt.title('E')

plt.subplot(2, 3, 4)
plt.plot(tls, u1)
plt.xlabel('Days')
plt.ylabel('Control')
plt.title(r'$u_1$')

plt.subplot(2, 3, 5)
plt.plot(tls, u2)
plt.xlabel('Days')
plt.ylabel('Control')
plt.title(r'$u_2$')

plt.subplot(2, 3, 6)
plt.plot(tls, u3)
plt.xlabel('Days')
plt.ylabel('Control')
plt.title(r'$u_3$')

plt.tight_layout()
